# Data-Aware Pocket Selection for Initializing Quantum DeepONets for PDEs

Reproducible notebook for the QC4PDE @ IEEE QCE 2026 workshop paper.

Implements two improvement directions over the first-moment barren-plateau framework
(arXiv:2606.18515) for a Quantum DeepONet (QONet):

1. **Data-averaged first-moment operator** &mdash; averages the Heisenberg-evolved readout over
   *both* the parameter-init ensemble and the PDE input-function ensemble, giving a computable
   data-averaged operator gap $D(P)=\tfrac1n\sum_k \mathbb{E}_{\theta}\,\mathrm{Var}_{u}[b_k(u;\theta)]$.
2. **Pocket-selection criterion** &mdash; a two-stage, training-free rule: a first-moment validity gate,
   then a pocket score $S(P)=\sqrt{D(P)\,T(P)}$ combining data alignment $D$ with trainability $T$.

Circuit: SU(2)/RY blocks + triangle-CX entangler + angle re-uploading, local $Z_k$ readout
(PennyLane `default.qubit`, JAX backprop). Operators: antiderivative, diffusion, nonlinear Burgers'.

Run **Kernel → Restart & Run All**. End-to-end runtime is a few minutes on CPU.

## 0. Environment

In [ ]:
# Run once; safe to skip if already installed.
%pip install -q pennylane jax jaxlib optax scipy matplotlib numpy

In [ ]:
import json, time
import numpy as np
import jax, jax.numpy as jnp, jax.random as jr
import pennylane as qml
import optax
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

# --- Computer Modern ("Modern Math") look for all figures ---
plt.rcParams.update({
    "mathtext.fontset": "cm",
    "font.family": "serif",
    "font.serif": ["cmr10", "CMU Serif", "DejaVu Serif"],
    "axes.formatter.use_mathtext": True,
    "axes.unicode_minus": False,
})

# --- global config ---
N      = 4    # qubits
LREUP  = 2    # data re-uploading encodings (theta has LREUP+1 SU1 blocks)
M      = 12   # query / sensor grid
NTR    = 40
NTE    = 30
print("jax", jax.__version__, "| pennylane", qml.__version__, "| optax", optax.__version__)

## 1. Data generation (PDE solution operators)

Antiderivative on a Chebyshev function space; diffusion (heat propagator) and **nonlinear**
Burgers' propagator on a cosine function space. Max-abs scaling, fixed query grid.

In [ ]:
def cheb_functions(ns, m, n_terms=5, rng=None):
    x = np.linspace(0., 1., m); xx = 2*x - 1
    out = np.zeros((ns, m))
    for s in range(ns):
        a = rng.uniform(-1, 1, n_terms)
        out[s] = np.polynomial.chebyshev.chebval(xx, a)
    return x, out

def cosine_functions(ns, m, rng=None):
    x = np.linspace(0., 1., m); out = np.zeros((ns, m))
    for s in range(ns):
        A = rng.uniform(0, 1); B = rng.uniform(-0.5, 0.5); P = rng.uniform(0, 2*np.pi)
        out[s] = A*np.cos(2*np.pi*x + P) + B
    return x, out

def antideriv(x, U):
    dx = x[1]-x[0]; S = np.zeros_like(U)
    S[:, 1:] = np.cumsum(0.5*(U[:, 1:]+U[:, :-1])*dx, axis=1)
    return S

def diffusion_step(x, U0, D=0.05, dt=0.05):
    m = x.shape[0]; k = 2*np.pi*np.fft.fftfreq(m, d=(x[1]-x[0]))
    decay = np.exp(-D*(k**2)*dt)
    return np.real(np.fft.ifft(np.fft.fft(U0, axis=1)*decay[None, :], axis=1))

def burgers_step(x, U0, nu=0.04, dt=0.04, nsub=300):
    m = x.shape[0]; k = 2*np.pi*np.fft.fftfreq(m, d=(x[1]-x[0]))
    u = U0.astype(float).copy(); h = dt/nsub
    for _ in range(nsub):
        uh = np.fft.fft(u, axis=1)
        ux = np.real(np.fft.ifft(1j*k[None, :]*uh, axis=1))
        uxx = np.real(np.fft.ifft(-(k[None, :]**2)*uh, axis=1))
        u = u + h*(-u*ux + nu*uxx)
    return u

def make_data(pde="antideriv"):
    rng = np.random.default_rng(0)
    if pde == "antideriv":
        x, U = cheb_functions(NTR+NTE, M, rng=rng); Y = antideriv(x, U)
    elif pde == "diffusion":
        x, U = cosine_functions(NTR+NTE, M, rng=rng); Y = diffusion_step(x, U)
    elif pde == "burgers":
        x, U = cosine_functions(NTR+NTE, M, rng=rng); Y = burgers_step(x, U)
    else:
        raise ValueError(pde)
    U = U/(np.abs(U).max()+1e-12); Y = Y/(np.abs(Y).max()+1e-12)
    return (jnp.array(x), jnp.array(U[:NTR]), jnp.array(Y[:NTR]),
            jnp.array(U[NTR:]), jnp.array(Y[NTR:]))

print("data fns ready")

## 2. QONet circuit (branch + trunk)

$\hat{\mathcal{G}}(u)(y)=\sum_k b_k(u)\,t_k(y)+b_0$ with $b_k=\langle B(u)|Z_k|B(u)\rangle$,
$t_k=\langle T(y)|Z_k|T(y)\rangle$. Each SU1 block = per-qubit RY then a fixed triangle-CX
entangler; data are re-uploaded via $R_Z(\pi/2 - \text{value})$.

In [ ]:
dev = qml.device("default.qubit", wires=N)

def _su1(theta_block):
    for q in range(N): qml.RY(theta_block[q], wires=q)
    for a in range(N):
        for b in range(a+1, N): qml.CNOT(wires=[a, b])   # triangle entangler

@qml.qnode(dev, interface="jax", diff_method="backprop")
def branch(theta, v):                 # theta (LREUP+1, N) ; v (M,)
    for l in range(LREUP):
        _su1(theta[l])
        for q in range(N): qml.RZ(jnp.pi/2 - v[(l*N+q) % M], wires=q)
    _su1(theta[LREUP])
    return [qml.expval(qml.PauliZ(q)) for q in range(N)]

@qml.qnode(dev, interface="jax", diff_method="backprop")
def trunk(theta, y):                  # theta (LREUP+1, N) ; y scalar
    for l in range(LREUP):
        _su1(theta[l])
        for q in range(N): qml.RZ(jnp.pi/2 - y*(q+1), wires=q)
    _su1(theta[LREUP])
    return [qml.expval(qml.PauliZ(q)) for q in range(N)]

bvec  = jax.jit(lambda th, v: jnp.stack(branch(th, v)))
tvec  = jax.jit(lambda th, y: jnp.stack(trunk(th, y)))
B_all = jax.jit(jax.vmap(bvec, in_axes=(None, 0)))   # (batch, N)
T_all = jax.jit(jax.vmap(tvec, in_axes=(None, 0)))   # (batch, N)
print("circuit ready")

## 3. First-moment initialization families

Each family is parameterized by its first moment $\mathbb{E}[\cos\theta]$ (the per-layer
retention $\kappa_\ell$). $\mathbb{E}[\cos]\!=\!0$ is the barren-plateau fixed point.

In [ ]:
SHP = (LREUP+1, N)
FAMILIES = {  # name: (sampler(key)->theta, E[cos theta], valid?)
    "identity N(0,.05)":    (lambda k: 0.05*jr.normal(k, SHP),                 float(np.exp(-0.05**2/2)), True),
    "small-Gauss N(0,.3)":  (lambda k: 0.30*jr.normal(k, SHP),                 float(np.exp(-0.30**2/2)), True),
    "calc_a U[-.4pi,.4pi]": (lambda k: jr.uniform(k, SHP, minval=-0.4*np.pi, maxval=0.4*np.pi), float(np.sin(0.4*np.pi)/(0.4*np.pi)), True),
    "shifted N(pi/4,.3)":   (lambda k: np.pi/4 + 0.30*jr.normal(k, SHP),       float(np.cos(np.pi/4)*np.exp(-0.30**2/2)), True),
    "BP U[-pi,pi]":         (lambda k: jr.uniform(k, SHP, minval=-np.pi, maxval=np.pi), 0.0, False),
}
for n_, (_, e_, v_) in FAMILIES.items():
    print(f"{n_:24s} E[cos]={e_:.3f}  valid={v_}")

## 4. Diagnostics: data-averaged gap $D$, trainability $T$, pocket score $S$

In [ ]:
def diagnostics(Utr, S=60):
    """T = Var over inits of d<Z_0>/d theta_1 ; D = data-averaged operator gap ; S = sqrt(D*T)."""
    dC = jax.jit(jax.grad(lambda th, v: bvec(th, v)[0]))
    rows = {}
    for name, (samp, ecos, valid) in FAMILIES.items():
        keys = jr.split(jr.key(7), S)
        gs = [float(dC(samp(keys[i]), Utr[i % Utr.shape[0]])[0, 0]) for i in range(S)]
        T = float(np.var(gs))
        ds = [float(jnp.mean(jnp.var(B_all(samp(keys[i]), Utr), axis=0))) for i in range(S)]
        D = float(np.mean(ds))
        rows[name] = dict(Ecos=ecos, valid=valid, T=T, D=D, S=float(np.sqrt(max(D,0)*max(T,0))))
    return rows

print("diagnostics() ready")

## 5. Stage-1 gate: first-moment contraction with depth

$\mathrm{Var}[\partial\langle Z_0\rangle/\partial\theta_1]$ vs re-uploading depth $L$ at fixed $n$.
Predicted $\sim \mathbb{E}[\cos]^{2L}$: the $\mathbb{E}[\cos]=0$ family collapses fastest.

In [ ]:
def branch_builder(n, lreup):
    d = qml.device("default.qubit", wires=n)
    def su1(tb):
        for q in range(n): qml.RY(tb[q], wires=q)
        for a in range(n):
            for b in range(a+1, n): qml.CNOT(wires=[a, b])
    @qml.qnode(d, interface="jax", diff_method="backprop")
    def circ(theta, v):
        for l in range(lreup):
            su1(theta[l])
            for q in range(n): qml.RZ(jnp.pi/2 - v[(l*n+q) % v.shape[0]], wires=q)
        su1(theta[lreup])
        return qml.expval(qml.PauliZ(0))
    return circ

def depth_scan(n0=6, depths=(2, 6, 10, 16, 22), S=40):
    fams = {"identity N(0,.05)": (lambda k, sh: 0.05*jr.normal(k, sh), float(np.exp(-0.05**2/2))),
            "small-Gauss N(0,.3)": (lambda k, sh: 0.30*jr.normal(k, sh), float(np.exp(-0.30**2/2))),
            "calc_a U[-.4pi,.4pi]": (lambda k, sh: jr.uniform(k, sh, minval=-0.4*np.pi, maxval=0.4*np.pi), float(np.sin(0.4*np.pi)/(0.4*np.pi))),
            "BP U[-pi,pi]": (lambda k, sh: jr.uniform(k, sh, minval=-np.pi, maxval=np.pi), 0.0)}
    out = {f: [] for f in fams}; ecos = {f: e for f, (s, e) in fams.items()}
    for L in depths:
        circ = branch_builder(n0, L)
        dC = jax.jit(jax.grad(lambda th, v: circ(th, v)))
        sh = (L+1, n0)
        for f, (samp, e) in fams.items():
            keys = jr.split(jr.key(3), 2*S)
            gs = [float(dC(samp(keys[i], sh), jr.uniform(keys[S+i], (n0,), minval=-1, maxval=1))[0, 0]) for i in range(S)]
            out[f].append(float(np.var(gs)))
    return {"depths": list(depths), "n": n0, "var": out, "ecos": ecos}

print("depth_scan() ready")

## 6. Training (Adam) and the validation loop

In [ ]:
def rel_l2(pred, tgt):
    return float(jnp.linalg.norm(pred-tgt)/(jnp.linalg.norm(tgt)+1e-12))

def train_family(name, seed, data, epochs=400):
    XQ, Utr, Ytr, Ute, Yte = data
    samp, ecos, valid = FAMILIES[name]
    k1, k2 = jr.split(jr.key(1000+seed))
    params = {"b": samp(k1), "t": samp(k2), "bias": jnp.array(0.0)}
    def predict(p, U, Y):
        return B_all(p["b"], U) @ T_all(p["t"], Y).T + p["bias"]
    def loss(p):
        return jnp.mean((predict(p, Utr, XQ) - Ytr)**2)
    opt = optax.adam(2e-2); st = opt.init(params)
    vg = jax.jit(jax.value_and_grad(loss))
    for _ in range(epochs):
        L, g = vg(params); u, st = opt.update(g, st); params = optax.apply_updates(params, u)
    return dict(family=name, seed=seed, Ecos=ecos, valid=valid,
                train_relL2=rel_l2(predict(params, Utr, XQ), Ytr),
                test_relL2=rel_l2(predict(params, Ute, XQ), Yte))

print("train_family() ready")

## 7. Run the diagnostics + depth scan (Table I, Figure 1a)

In [ ]:
data_ad = make_data("antideriv")
diag = diagnostics(data_ad[1])
print(f"{'family':24s}{'E[cos]':>8s}{'valid':>7s}{'T(train)':>12s}{'D(data-gap)':>13s}{'S(pocket)':>12s}")
for n_, r in diag.items():
    print(f"{n_:24s}{r['Ecos']:8.3f}{str(r['valid']):>7s}{r['T']:12.3e}{r['D']:13.3e}{r['S']:12.3e}")

dep = depth_scan()
print("\nVar[grad] vs depth (n=%d):" % dep["n"])
for f, vs in dep["var"].items():
    print(f"{f:24s} E[cos]={dep['ecos'][f]:.3f}  " + " ".join(f"{v:9.2e}" for v in vs))

## 8. Train all families on all three operators (Table II)

$D,T,S$ depend only on the branch + input ensemble (training-free, target-agnostic);
diffusion and Burgers' share inputs, hence one $S$ column. 5 seeds each.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
results = {}
for pde in ["antideriv", "diffusion", "burgers"]:
    data = make_data(pde)
    dg = diagnostics(data[1])
    out = {}
    t0 = time.time()
    for fam in FAMILIES:
        tes = [train_family(fam, s, data)["test_relL2"] for s in SEEDS]
        d = dg[fam]
        out[fam] = dict(Ecos=d["Ecos"], valid=d["valid"], T=d["T"], D=d["D"], S=d["S"],
                        test_mean=float(np.mean(tes)), test_std=float(np.std(tes)), test_all=tes)
        print(f"[{pde:9s}] {fam:22s} valid={str(d['valid']):>5s} S={d['S']:.3e} "
              f"testRMSE={np.mean(tes):.4f}+/-{np.std(tes):.4f}")
    results[pde] = out
    print(f"  -> {pde} done in {time.time()-t0:.1f}s\n")

for pde in results:
    sx = [v["S"] for v in results[pde].values() if v["valid"]]
    ey = [v["test_mean"] for v in results[pde].values() if v["valid"]]
    rho, _ = spearmanr(sx, ey)
    print(f"Spearman(S, test error) over valid families [{pde}] = {rho:.3f}")

## 9. Figure 1 — Stage-1 gate (a) and Stage-2 criterion (b)

In [ ]:
res = results["antideriv"]
short = {"identity N(0,.05)":"identity","small-Gauss N(0,.3)":"small-Gauss",
         "calc_a U[-.4pi,.4pi]":"calc-a","shifted N(pi/4,.3)":"shifted","BP U[-pi,pi]":"BP-uniform"}
col   = {"identity N(0,.05)":"#1f77b4","small-Gauss N(0,.3)":"#2ca02c",
         "calc_a U[-.4pi,.4pi]":"#ff7f0e","shifted N(pi/4,.3)":"#9467bd","BP U[-pi,pi]":"#d62728"}

# ============ figure style knobs (edit these) ============
FIG_W, FIG_H = 12, 6.2        # figure size in inches
TITLE_FS   = 17               # subplot-title font size
LABEL_FS   = 15               # axis-label font size
TICK_FS    = 13               # tick-number font size
LEGEND_FS  = 12               # legend font size
ANNOT_FS   = 11               # point-label font size
MARK_MS    = 14               # marker size in panel (b)
LEG_LOC_A  = "lower left"     # legend location, panel (a)
LEG_LOC_B  = "upper right"    # legend location, panel (b)
LEG_BBOX_A = None             # e.g. (1.01, 1.0) to push the box outside the axes; None = off
LEG_BBOX_B = None             # e.g. (1.01, 1.0); None = off
# =========================================================

fig, ax = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))
# (a) first-moment depth contraction
for f, vs in dep["var"].items():
    ax[0].semilogy(dep["depths"], vs, marker="o", ms=5, color=col[f],
                   label=f"{short[f]} (E[cos]={dep['ecos'][f]:.2f})")
ax[0].set_xlabel("re-uploading depth L", fontsize=LABEL_FS)
ax[0].set_ylabel(r"Var$[\partial\langle Z_0\rangle/\partial\theta_1]$", fontsize=LABEL_FS)
ax[0].set_title(f"(a) Stage-1 gate: first-moment contraction (n={dep['n']})", fontsize=TITLE_FS)
ax[0].tick_params(axis="both", labelsize=TICK_FS)
ax[0].grid(True, which="both", alpha=0.3)
ax[0].legend(fontsize=LEGEND_FS, loc=LEG_LOC_A, bbox_to_anchor=LEG_BBOX_A)
# (b) pocket score predicts error
LBL = {"identity N(0,.05)": (10, 4, "left"), "small-Gauss N(0,.3)": (10, 4, "left"),
       "shifted N(pi/4,.3)": (-9, -3, "right"), "calc_a U[-.4pi,.4pi]": (-12, 15, "right"),
       "BP U[-pi,pi]": (10, 17, "left")}
for f, r in res.items():
    mk = "o" if r["valid"] else "X"
    ax[1].errorbar(r["S"], r["test_mean"], yerr=r["test_std"], marker=mk, ms=MARK_MS, capsize=6,
                   color=col[f], mfc=(col[f] if r["valid"] else "none"), mew=2,
                   label=short[f] + ("" if r["valid"] else " (gated out)"))
    dx, dy, ha = LBL[f]
    ax[1].annotate(short[f] + ("" if r["valid"] else " (gated out)"), (r["S"], r["test_mean"]),
                   textcoords="offset points", xytext=(dx, dy), ha=ha, fontsize=ANNOT_FS)
valid = sorted((r["S"], r["test_mean"]) for r in res.values() if r["valid"])
ax[1].plot(*zip(*valid), "--", color="gray", lw=1, alpha=0.7, zorder=0)
ax[1].set_xscale("log"); ax[1].set_xlim(2e-5, 1.6e-1)
ax[1].set_xlabel(r"pocket score  $S=(D\cdot T)^{1/2}$", fontsize=LABEL_FS)
ax[1].set_ylabel(r"test relative $L_2$ error", fontsize=LABEL_FS)
ax[1].set_title("(b) Stage-2 criterion: $S$ predicts downstream error", fontsize=TITLE_FS)
ax[1].tick_params(axis="both", labelsize=TICK_FS)
ax[1].grid(True, which="both", alpha=0.3)
ax[1].legend(fontsize=LEGEND_FS, loc=LEG_LOC_B, bbox_to_anchor=LEG_BBOX_B)
plt.tight_layout(); plt.savefig("pocket_selection.png", dpi=300, bbox_inches="tight"); plt.show()

## 10. Figure 2 — Generalization across solution operators

In [ ]:
pdes = {"antideriv": ("Antiderivative (linear)", "o"),
        "diffusion": ("Diffusion (linear)", "s"),
        "burgers":   ("Burgers' (nonlinear)", "^")}
pcol = {"antideriv": "#1f77b4", "diffusion": "#2ca02c", "burgers": "#d62728"}

# ============ figure style knobs (edit these) ============
FIG_W, FIG_H = 6.2, 4.4        # figure size in inches
TITLE_FS     = 13             # title font size
LABEL_FS     = 12             # axis-label font size
TICK_FS      = 11             # tick-number font size
LEGEND_FS    = 10             # legend entry font size
LEGTITLE_FS  = 9             # legend-title (the rho values) font size
MARK_MS      = 9             # marker size
LEG_LOC      = "lower left"   # legend location: 'best','upper right','lower left',...
LEG_BBOX     = None           # e.g. (1.01, 1.0) to push the box outside the axes; None = off
# =========================================================

fig2, ax2 = plt.subplots(figsize=(FIG_W, FIG_H)); rho_txt = []
for pde, (lbl, mk) in pdes.items():
    rr = results[pde]
    sx = [v["S"] for v in rr.values() if v["valid"]]
    ey = [v["test_mean"] for v in rr.values() if v["valid"]]
    es = [v["test_std"] for v in rr.values() if v["valid"]]
    o = np.argsort(sx); sx = np.array(sx)[o]; ey = np.array(ey)[o]; es = np.array(es)[o]
    ax2.errorbar(sx, ey, yerr=es, marker=mk, ms=MARK_MS, capsize=3, lw=1.3, color=pcol[pde], label=lbl)
    rho_p, _ = spearmanr(sx, ey); rho_txt.append(f"{lbl.split()[0]}: $\\rho$={rho_p:.2f}")
ax2.set_xscale("log")
ax2.set_xlabel(r"pocket score  $S=(D\cdot T)^{1/2}$  (training-free, computed from inputs)", fontsize=LABEL_FS)
ax2.set_ylabel(r"test relative $L_2$ error", fontsize=LABEL_FS)
ax2.set_title("Pocket score predicts error across three solution operators", fontsize=TITLE_FS)
ax2.tick_params(axis="both", labelsize=TICK_FS)
ax2.grid(True, which="both", alpha=0.3)
ax2.legend(title="  ".join(rho_txt), fontsize=LEGEND_FS, title_fontsize=LEGTITLE_FS,
           loc=LEG_LOC, bbox_to_anchor=LEG_BBOX)
plt.tight_layout(); plt.savefig("pocket_generalization.png", dpi=300, bbox_inches="tight"); plt.show()

## 11. Save numeric results (optional)

In [ ]:
for pde in results:
    json.dump(results[pde], open(f"results_{pde}.json", "w"), indent=2)
json.dump(dep, open("depth.json", "w"), indent=2)
print("saved results_*.json, depth.json, pocket_selection.png, pocket_generalization.png")